# Experiment 7: regularized time trend + AutoGluon residuals

This experiment gives the model an explicitly extrapolative trend: a Ridge model learns a global log-rent slope plus regularized `town × flat-type` slope deviations. AutoGluon then learns property-level residuals.

Validation is strictly forward-looking in two forms: (1) expanding-window rolling-origin cross-validation with six-month horizons and (2) calendar-year holdouts. Random folds are intentionally not used because the competition test set is later in time.

In [1]:
# If needed: pip install autogluon.tabular
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
candidate_roots = ([Path(configured_root)] if configured_root else []) + [
    Path.cwd(), *Path.cwd().parents, Path('/content/KDDM-project'),
    Path('/content/drive/MyDrive/KDDM-project'),
]
project_root = next((p for p in candidate_roots if (p / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))

from autogluon.tabular import TabularPredictor
from preprocessing import RentalPreprocessor, load_datasets

RANDOM_STATE = 42
TARGET = 'MONTHLY_RENT'
ROLLING_HORIZON_MONTHS = 6
ROLLING_STEP_MONTHS = 6
MIN_HISTORY_MONTHS = 12
TREND_ALPHA = 30.0  # Higher values shrink segment-specific slopes toward the national slope.
MODEL_TIME_LIMIT = int(os.environ.get('EXP7_MODEL_TIME_LIMIT', '180'))
AG_HYPERPARAMETERS = {'GBM': {}, 'CAT': {}}


In [2]:
def canonical_flat_type(values):
    return values.fillna('unknown').astype(str).str.lower().str.replace(r'[^a-z0-9]+', '', regex=True)

def add_time_keys(frame, row_offset=0):
    result = frame.copy().reset_index(drop=True)
    result['_row_id'] = np.arange(row_offset, row_offset + len(result))
    result['period'] = pd.to_datetime(result['RENT_APPROVAL_DATE'] + '-01')
    town = result['TOWN'].fillna('unknown').astype(str).str.strip().str.lower()
    result['market_segment'] = town + ' | ' + canonical_flat_type(result['FLAT_TYPE'])
    return result

def build_monthly_covariates(datasets):
    month_number = {name: number for number, name in enumerate(
        ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'], 1
    )}
    coe = datasets['coe'].copy()
    coe['period'] = pd.to_datetime(dict(
        year=coe['YEAR'], month=coe['MONTH'].str.lower().map(month_number), day=1
    ))
    for column in ['QUOTA_PREMIUM', 'BIDS_RECEIVED']:
        coe[column] = pd.to_numeric(
            coe[column].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False),
            errors='coerce',
        )
    result = coe.groupby('period', as_index=False).agg(
        coe_quota_premium_mean=('QUOTA_PREMIUM', 'mean'),
        coe_bids_mean=('BIDS_RECEIVED', 'mean'),
    )
    result['month_sin'] = np.sin(2 * np.pi * result['period'].dt.month / 12)
    result['month_cos'] = np.cos(2 * np.pi * result['period'].dt.month / 12)
    return result

data_dir, datasets = load_datasets(project_root / 'data')
raw_train = add_time_keys(datasets['train'])
raw_test = add_time_keys(datasets['test'])
monthly_covariates = build_monthly_covariates(datasets)
time_origin = raw_train['period'].min()
forecast_origin = raw_test['period'].min()

def trend_features(frame):
    result = frame.merge(monthly_covariates, on='period', how='left', validate='many_to_one').copy()
    result['time_index'] = (result['period'].dt.year - time_origin.year) * 12 + result['period'].dt.month - time_origin.month
    needed = ['time_index', 'month_sin', 'month_cos', 'coe_bids_mean', 'coe_quota_premium_mean']
    if result[needed].isna().any().any():
        raise ValueError(f'Missing trend covariates: {result[needed].isna().sum().to_dict()}')
    return result

# Strictly available history: labelled rents in the first test month are excluded.
history_train = raw_train[raw_train['period'] < forecast_origin].copy()
print(f'Labelled history: {history_train.period.min():%Y-%m} to {history_train.period.max():%Y-%m}')
print(f'Test forecast period: {raw_test.period.min():%Y-%m} to {raw_test.period.max():%Y-%m}')


Labelled history: 2021-01 to 2025-02
Test forecast period: 2025-03 to 2026-07


## Trend model

The design matrix contains a global intercept and time slope, a one-hot segment intercept, and a segment × time interaction. Ridge regularization keeps a small segment from extrapolating an implausibly steep slope. COE bids, COE premium, and cyclic month features are added only because the auxiliary file supplies them throughout the test horizon.

In [3]:
TREND_NUMERIC_COLUMNS = ['time_index', 'month_sin', 'month_cos', 'coe_bids_mean', 'coe_quota_premium_mean']

class ShrunkSegmentTrend:
    """Log-rent trend with a global slope and Ridge-shrunk segment slopes."""
    def __init__(self, alpha=TREND_ALPHA):
        self.alpha = alpha

    def _new_encoder(self):
        try:
            return OneHotEncoder(handle_unknown='ignore', sparse_output=True)
        except TypeError:  # scikit-learn < 1.2
            return OneHotEncoder(handle_unknown='ignore', sparse=True)

    def _matrix(self, frame, fit=False):
        features = trend_features(frame)
        if fit:
            self.segment_encoder_ = self._new_encoder()
            segment = self.segment_encoder_.fit_transform(features[['market_segment']])
            self.numeric_scaler_ = StandardScaler()
            numeric = self.numeric_scaler_.fit_transform(features[TREND_NUMERIC_COLUMNS])
        else:
            segment = self.segment_encoder_.transform(features[['market_segment']])
            numeric = self.numeric_scaler_.transform(features[TREND_NUMERIC_COLUMNS])
        # The first scaled numeric column is time, so this learns a separate, regularized slope per segment.
        segment_time = segment.multiply(numeric[:, [0]])
        return sparse.hstack([sparse.csr_matrix(numeric), segment, segment_time], format='csr')

    def fit(self, frame):
        matrix = self._matrix(frame, fit=True)
        self.model_ = Ridge(alpha=self.alpha, fit_intercept=True, solver='lsqr', random_state=RANDOM_STATE)
        self.model_.fit(matrix, np.log(frame[TARGET].astype(float)))
        return self

    def predict_log_rent(self, frame):
        return self.model_.predict(self._matrix(frame, fit=False))

def rmse(actual, prediction):
    return mean_squared_error(actual, prediction) ** 0.5


In [4]:
def rolling_origin_splits(frame):
    months = pd.DatetimeIndex(sorted(frame['period'].unique()))
    splits = []
    for start_index in range(MIN_HISTORY_MONTHS, len(months), ROLLING_STEP_MONTHS):
        start = months[start_index]
        end = start + pd.DateOffset(months=ROLLING_HORIZON_MONTHS)
        train = frame[frame['period'] < start].copy()
        valid = frame[(frame['period'] >= start) & (frame['period'] < end)].copy()
        expected_months = pd.date_range(start, periods=ROLLING_HORIZON_MONTHS, freq='MS')
        if len(train) and set(expected_months).issubset(set(valid['period'].unique())):
            splits.append((f'rolling_{start:%Y_%m}', train, valid))
    return splits

def calendar_year_splits(frame):
    splits = []
    for year in sorted(frame['period'].dt.year.unique()):
        start = pd.Timestamp(year=year, month=1, day=1)
        train = frame[frame['period'] < start].copy()
        valid = frame[frame['period'].dt.year == year].copy()
        # Require a complete calendar year; 2025 has only Jan/Feb before the strict forecast origin.
        if len(train) and valid['period'].nunique() == 12:
            splits.append((f'year_{year}', train, valid))
    return splits

rolling_splits = rolling_origin_splits(history_train)
year_splits = calendar_year_splits(history_train)
print('Rolling-origin folds:', [name for name, _, _ in rolling_splits])
print('Calendar-year folds:', [name for name, _, _ in year_splits])


Rolling-origin folds: ['rolling_2022_01', 'rolling_2022_07', 'rolling_2023_01', 'rolling_2023_07', 'rolling_2024_01', 'rolling_2024_07']
Calendar-year folds: ['year_2022', 'year_2023', 'year_2024']


In [5]:
def prepare_tabular_features(train_frame, valid_frame):
    preprocessor = RentalPreprocessor(datasets)
    train_processed = preprocessor.fit_transform(train_frame).sort_values('_row_id')
    valid_processed = preprocessor.transform(valid_frame).sort_values('_row_id')

    def clean(frame):
        result = frame.drop(columns=[TARGET, '_row_id', 'period', 'market_segment'], errors='ignore').copy()
        result['RENT_APPROVAL_DATE'] = pd.to_datetime(result['RENT_APPROVAL_DATE'] + '-01')
        return result

    return clean(train_processed), clean(valid_processed)

def fit_tabular_predictor(label, train_x, train_y, path):
    return TabularPredictor(
        label=label, problem_type='regression', eval_metric='root_mean_squared_error', path=str(path),
    ).fit(
        train_data=train_x.assign(**{label: train_y}),
        hyperparameters=AG_HYPERPARAMETERS, presets='medium_quality',
        time_limit=MODEL_TIME_LIMIT, verbosity=1,
    )

def evaluate_split(split_name, train_frame, valid_frame, scheme):
    train_frame = train_frame.sort_values('_row_id').copy()
    valid_frame = valid_frame.sort_values('_row_id').copy()
    trend = ShrunkSegmentTrend().fit(train_frame)
    trend_train_log = trend.predict_log_rent(train_frame)
    trend_valid_log = trend.predict_log_rent(valid_frame)
    residual_train = np.log(train_frame[TARGET].to_numpy()) - trend_train_log

    train_x, valid_x = prepare_tabular_features(train_frame, valid_frame)
    residual_predictor = fit_tabular_predictor(
        'LOG_RENT_RESIDUAL', train_x, residual_train,
        experiment_dir / 'ag_exp7' / split_name / 'residual',
    )
    baseline_predictor = fit_tabular_predictor(
        TARGET, train_x, train_frame[TARGET].to_numpy(),
        experiment_dir / 'ag_exp7' / split_name / 'baseline',
    )

    trend_prediction = np.exp(trend_valid_log)
    residual_prediction = residual_predictor.predict(valid_x).to_numpy()
    hybrid_prediction = trend_prediction * np.exp(residual_prediction)
    baseline_prediction = baseline_predictor.predict(valid_x).to_numpy()
    actual = valid_frame.sort_values('_row_id')[TARGET].to_numpy()

    rows = pd.DataFrame({
        'scheme': scheme, 'split': split_name, 'row_id': valid_frame.sort_values('_row_id')['_row_id'].to_numpy(),
        'period': valid_frame.sort_values('_row_id')['period'].to_numpy(), 'actual': actual,
        'trend_prediction': trend_prediction, 'baseline_prediction': baseline_prediction,
        'hybrid_prediction': hybrid_prediction,
    })
    metrics = pd.DataFrame([
        {'scheme': scheme, 'split': split_name, 'model': 'trend_only', 'RMSE': rmse(actual, trend_prediction), 'n_rows': len(rows)},
        {'scheme': scheme, 'split': split_name, 'model': 'autogluon_baseline', 'RMSE': rmse(actual, baseline_prediction), 'n_rows': len(rows)},
        {'scheme': scheme, 'split': split_name, 'model': 'trend_plus_residual', 'RMSE': rmse(actual, hybrid_prediction), 'n_rows': len(rows)},
    ])
    return metrics, rows

all_metrics, all_predictions = [], []
for scheme, splits in [('rolling_origin', rolling_splits), ('calendar_year', year_splits)]:
    for split_name, train_fold, valid_fold in splits:
        print(f'Running {split_name}: train through {train_fold.period.max():%Y-%m}, validate {valid_fold.period.min():%Y-%m} to {valid_fold.period.max():%Y-%m}')
        metrics, predictions = evaluate_split(split_name, train_fold, valid_fold, scheme)
        all_metrics.append(metrics)
        all_predictions.append(predictions)

split_metrics = pd.concat(all_metrics, ignore_index=True)
oof_predictions = pd.concat(all_predictions, ignore_index=True)
split_metrics_path = experiment_dir / 'exp_7_split_rmse.csv'
oof_path = experiment_dir / 'exp_7_oof_predictions.csv'
split_metrics.to_csv(split_metrics_path, index=False)
oof_predictions.to_csv(oof_path, index=False)
display(split_metrics.pivot(index=['scheme', 'split'], columns='model', values='RMSE').round(3))
print(f'Saved split-level RMSE to {split_metrics_path}')


Running rolling_2022_01: train through 2021-12, validate 2022-01 to 2022-06


Running rolling_2022_07: train through 2022-06, validate 2022-07 to 2022-12


Running rolling_2023_01: train through 2022-12, validate 2023-01 to 2023-06


Running rolling_2023_07: train through 2023-06, validate 2023-07 to 2023-12
Running rolling_2024_01: train through 2023-12, validate 2024-01 to 2024-06


Running rolling_2024_07: train through 2024-06, validate 2024-07 to 2024-12


Running year_2022: train through 2021-12, validate 2022-01 to 2022-12


Running year_2023: train through 2022-12, validate 2023-01 to 2023-12


Running year_2024: train through 2023-12, validate 2024-01 to 2024-12


model                           autogluon_baseline  trend_only  \
scheme         split                                             
calendar_year  year_2022                   676.074     609.240   
               year_2023                   651.873     666.842   
               year_2024                   504.211     557.890   
rolling_origin rolling_2022_01             475.916     451.098   
               rolling_2022_07             660.882     689.758   
               rolling_2023_01             665.617     676.437   
               rolling_2023_07             589.179     684.049   
               rolling_2024_01             524.614     557.812   
               rolling_2024_07             464.260     508.028   

model                           trend_plus_residual  
scheme         split                                 
calendar_year  year_2022                    641.148  
               year_2023                    645.255  
               year_2024                    528.011  
rolling_origin rolling_2022_01              471.936  
               rolling_2022_07              646.509  
               rolling_2023_01              634.166  
               rolling_2023_07              643.622  
               rolling_2024_01              525.926  
               rolling_2024_07              484.923

Saved split-level RMSE to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_7_split_rmse.csv


In [6]:
def pooled_rmse(rows, prediction_column):
    return rmse(rows['actual'], rows[prediction_column])

summary_rows = []
for scheme, rows in oof_predictions.groupby('scheme'):
    for model, column in {
        'trend_only': 'trend_prediction',
        'autogluon_baseline': 'baseline_prediction',
        'trend_plus_residual': 'hybrid_prediction',
    }.items():
        summary_rows.append({
            'scheme': scheme, 'model': model, 'pooled_RMSE': pooled_rmse(rows, column),
            'mean_fold_RMSE': split_metrics.query('scheme == @scheme and model == @model')['RMSE'].mean(),
            'n_rows': len(rows),
        })
rmse_summary = pd.DataFrame(summary_rows).sort_values(['scheme', 'pooled_RMSE'])
summary_path = experiment_dir / 'exp_7_rmse_summary.csv'
rmse_summary.to_csv(summary_path, index=False)
display(rmse_summary.round(3))

# Select the production model using only pooled rolling-origin out-of-fold performance.
rolling_summary = rmse_summary[rmse_summary['scheme'] == 'rolling_origin'].set_index('model')
production_model = rolling_summary['pooled_RMSE'].idxmin()
print(f'Production choice from rolling-origin CV: {production_model}')
print(f'Saved summary RMSE to {summary_path}')


,scheme,model,pooled_RMSE,mean_fold_RMSE,n_rows
2,calendar_year,trend_plus_residual,606.495,604.805,103457
0,calendar_year,trend_only,613.263,611.324,103457
1,calendar_year,autogluon_baseline,613.928,610.719,103457
4,rolling_origin,autogluon_baseline,567.562,563.411,103457
5,rolling_origin,trend_plus_residual,571.673,567.847,103457
3,rolling_origin,trend_only,600.585,594.530,103457


Production choice from rolling-origin CV: autogluon_baseline
Saved summary RMSE to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_7_rmse_summary.csv


In [7]:
# Refit both models on all history before the first test month, then predict the future test period.
final_history = history_train.sort_values('_row_id').copy()
final_test = raw_test.sort_values('_row_id').copy()
final_trend = ShrunkSegmentTrend().fit(final_history)
final_train_x, final_test_x = prepare_tabular_features(final_history, final_test)
final_train_trend_log = final_trend.predict_log_rent(final_history)
final_residual = np.log(final_history[TARGET].to_numpy()) - final_train_trend_log

final_residual_predictor = TabularPredictor(
    label='LOG_RENT_RESIDUAL', problem_type='regression', eval_metric='root_mean_squared_error',
    path=str(experiment_dir / 'ag_exp7' / 'final_residual'),
).fit(
    train_data=final_train_x.assign(LOG_RENT_RESIDUAL=final_residual),
    hyperparameters=AG_HYPERPARAMETERS, presets='medium_quality', time_limit=MODEL_TIME_LIMIT, verbosity=1,
)
final_baseline_predictor = TabularPredictor(
    label=TARGET, problem_type='regression', eval_metric='root_mean_squared_error',
    path=str(experiment_dir / 'ag_exp7' / 'final_baseline'),
).fit(
    train_data=final_train_x.assign(**{TARGET: final_history[TARGET].to_numpy()}),
    hyperparameters=AG_HYPERPARAMETERS, presets='medium_quality', time_limit=MODEL_TIME_LIMIT, verbosity=1,
)

final_test_trend_log = final_trend.predict_log_rent(final_test)
final_test_trend = np.exp(final_test_trend_log)
final_test_residual = final_residual_predictor.predict(final_test_x).to_numpy()
final_test_hybrid = final_test_trend * np.exp(final_test_residual)
final_test_baseline = final_baseline_predictor.predict(final_test_x).to_numpy()
prediction_by_model = {
    'trend_only': final_test_trend,
    'autogluon_baseline': final_test_baseline,
    'trend_plus_residual': final_test_hybrid,
}
final_prediction = np.asarray(prediction_by_model[production_model]).clip(0, None)

diagnostic_output = final_test.drop(columns=['_row_id', 'period', 'market_segment']).copy()
diagnostic_output['TREND_ONLY_PREDICTION'] = final_test_trend
diagnostic_output['AUTOGLOUON_BASELINE_PREDICTION'] = final_test_baseline
diagnostic_output['TREND_PLUS_RESIDUAL_PREDICTION'] = final_test_hybrid
diagnostic_output['PREDICTED_MONTHLY_RENT'] = final_prediction
diagnostic_path = experiment_dir / 'exp_7_trend_residual_predictions.csv'
diagnostic_output.to_csv(diagnostic_path, index=False)

competition_submission = pd.DataFrame({'Id': final_test['_row_id'].to_numpy(), 'Predicted': final_prediction})
assert competition_submission['Id'].is_unique and competition_submission['Predicted'].notna().all()
submission_path = experiment_dir / 'exp_7_trend_residual_submission.csv'
competition_submission.to_csv(submission_path, index=False)
print(f'Saved diagnostics to {diagnostic_path}')
print(f'Saved {len(competition_submission):,} competition predictions to {submission_path}')
display(competition_submission.head())


Saved diagnostics to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_7_trend_residual_predictions.csv
Saved 50,000 competition predictions to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_7_trend_residual_submission.csv


,Id,Predicted
0,0,2729.248535
1,1,3697.990479
2,2,3145.001465
3,3,3191.828857
4,4,3291.430176
